# Train a Wordle agent with OpenEnv: SFT, then GRPO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huggingface/OpenEnv/blob/main/examples/wordle/wordle.ipynb)

This notebook follows the [OpenEnv Wordle tutorial](https://huggingface.co/docs/openenv/tutorials/wordle-grpo): Qwen3-1.7B can't win a game of Wordle, a short SFT warmup teaches it the game, and GRPO against the [`openenv/wordle`](https://huggingface.co/spaces/openenv/wordle) environment cuts its mistakes.

- **On a free T4** you can play the environment, generate the SFT data, launch SFT and GRPO on [Hugging Face Jobs](https://huggingface.co/docs/huggingface_hub/guides/jobs) and compare the published models.
- **On an A100 (Colab Pro)** you can also run SFT and GRPO in the notebook itself.

In [ ]:
!pip install -q "openenv==0.10.0" "openenv-textarena @ git+https://github.com/huggingface/OpenEnv.git@v0.10.0#subdirectory=envs/textarena_env" "trl==1.16.0" trackio

Log in to Hugging Face to push datasets and models and to launch Jobs. You need a token with write access.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 1. Play the environment

The environment runs as a Space. Each guess returns the feedback for every letter: `G` (right letter, right spot), `Y` (in the word, wrong spot) and `X` (not in the word).

In [ ]:
from textarena_env import TextArenaAction, TextArenaEnv

with TextArenaEnv(base_url="https://openenv-wordle.hf.space").sync() as env:
    result = env.reset()
    print(result.observation.messages[0].content)
    result = env.step(TextArenaAction(message="[crane]"))
    print(result.observation.messages[0].content[-200:])

## 2. Generate the SFT data

A solver plays 1,000 games against the same environment class the Space runs (in-process, so it takes seconds) and writes each one as a tool-calling conversation, with a short reasoning before every guess. Replace `<user>` with your Hugging Face username.

In [ ]:
!wget -q https://raw.githubusercontent.com/huggingface/OpenEnv/main/examples/wordle/generate_sft_data.py https://raw.githubusercontent.com/huggingface/OpenEnv/main/examples/wordle/sft.py https://raw.githubusercontent.com/huggingface/OpenEnv/main/examples/wordle/grpo.py https://raw.githubusercontent.com/huggingface/OpenEnv/main/examples/wordle/evaluate.py https://raw.githubusercontent.com/huggingface/OpenEnv/main/examples/wordle/make_gif.py https://raw.githubusercontent.com/huggingface/OpenEnv/main/examples/wordle/secrets30.txt
!python generate_sft_data.py --games 1000 --push-to-hub <user>/wordle-tool-calls-sft

## 3. SFT and GRPO on Hugging Face Jobs

Both steps run on one A100 80GB (`a100-large`, $2.50/h): the SFT warmup takes {{SFT_TIME}} and GRPO {{GRPO_TIME}}, about {{COST}} in total.

GRPO plays 64 games at once, one environment session each. [Duplicate the `openenv/wordle` Space](https://huggingface.co/spaces/openenv/wordle?duplicate=true), set the `MAX_CONCURRENT_ENVS=64` variable on your copy and pass its URL as `--env-url`.

In [ ]:
!hf jobs uv run --flavor a100-large --timeout 1h --secrets HF_TOKEN sft.py \
    --dataset <user>/wordle-tool-calls-sft --hub-model-id <user>/Qwen3-1.7B-Wordle-SFT

Once the SFT job finishes, start GRPO from the SFT model. Follow the run with `hf jobs logs <job-id>` and on Trackio.

In [ ]:
!hf jobs uv run --flavor a100-large --timeout 4h --secrets HF_TOKEN grpo.py \
    --model <user>/Qwen3-1.7B-Wordle-SFT --hub-model-id <user>/Qwen3-1.7B-Wordle-SFT-GRPO \
    --env-url https://<user>-wordle.hf.space

### Or train in the notebook (A100)

With an A100 runtime, run the same scripts here instead. GRPO needs vLLM.

In [ ]:
# !pip install -q "trl[vllm]==1.16.0"
# !python sft.py --dataset <user>/wordle-tool-calls-sft --hub-model-id <user>/Qwen3-1.7B-Wordle-SFT
# !python grpo.py --model <user>/Qwen3-1.7B-Wordle-SFT --hub-model-id <user>/Qwen3-1.7B-Wordle-SFT-GRPO --env-url https://<user>-wordle.hf.space

## 4. Compare the models

`evaluate.py` plays a model greedily and saves every game. `--secrets` runs the environment in-process with fixed secret words, so every model plays the same ones. This uses the published models: swap in yours once they're trained. On a T4, each model takes a few minutes for 10 words.

In [ ]:
!python evaluate.py --model Qwen/Qwen3-1.7B --max-new-tokens 256 --secrets "$(head -c 59 secrets30.txt)" --out base.jsonl
!python evaluate.py --model openenv/Qwen3-1.7B-Wordle-SFT --thinking --max-new-tokens 512 --secrets "$(head -c 59 secrets30.txt)" --out sft.jsonl
!python evaluate.py --model openenv/Qwen3-1.7B-Wordle-SFT-GRPO --thinking --max-new-tokens 512 --secrets "$(head -c 59 secrets30.txt)" --out grpo.jsonl

In [ ]:
!pip install -q pillow
!python make_gif.py base.jsonl sft.jsonl grpo.jsonl --labels "Qwen3-1.7B" "+ SFT" "+ SFT + GRPO" --out progress.gif

from IPython.display import Image

Image("progress.gif")